# 7. LangSmith を使った RAG アプリケーションの評価


In [1]:
!pip install numpy==1.26.4

In [2]:
# 【注意】
# 上記の `!pip install numpy==1.26.4` を実行したあと、
# Google Colab 上部のメニューから「ランタイム」の「セッションを再起動する」を実行してください。
# その後このセルを実行して `1.26.4` と表示されることを確認してください。

import numpy as np

print(np.__version__)
assert np.__version__ == "1.26.4"

1.26.4


In [3]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "agent-book"

## 7.4. Ragas による合成テストデータの生成


### パッケージのインストール


In [4]:
!pip install langchain-core==0.2.30 langchain-openai==0.1.21 \
    langchain-community==0.2.12 GitPython==3.1.43 \
    langchain-chroma==0.1.2 chromadb==0.5.3 \
    ragas==0.1.14 nest-asyncio==1.6.0 pydantic==2.10.6

### 検索対象のドキュメントのロード


In [5]:
from langchain_community.document_loaders import GitLoader


def file_filter(file_path: str) -> bool:
    return file_path.endswith(".mdx")


loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="langchain==0.2.13",
    file_filter=file_filter,
)

documents = loader.load()
print(len(documents))

280


### Ragas による合成テストデータ生成の実装


In [6]:
for document in documents:
    document.metadata["filename"] = document.metadata["source"]

#### 【注意】既知のエラーについて

以下のコードで gpt-4o を使用すると OpenAI API の Usage tier 次第で RateLimitError が発生することが報告されています。

OpenAI API の Usage tier については公式ドキュメントの以下のページを参照してください。

https://platform.openai.com/docs/guides/rate-limits/usage-tiers

このエラーが発生した場合は、以下のどちらかの対応を実施してください。

1. 同じ Tier でも gpt-4o よりレートリミットの高い gpt-4o-mini を使用する
   - この場合、生成される合成テストデータの品質は低くなることが想定されます
2. 課金などにより Tier を上げる
   - Tier 2 で RateLimitError が発生しないことを確認済みです (2024 年 10 月 31 日時点)

##### 2025/3/15 追記

LangChain のドキュメントの増加により、gpt-4o-mini を使用しても Tier 1 ではエラーが発生することが報告されています。

その場合、上部のコードの GitHub からドキュメントをロードする箇所で、以下のように `langchain==0.2.13` という動作確認済みのバージョンを指定するようにしてください。

```python
loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="langchain==0.2.13",
    file_filter=file_filter,
)
```


In [8]:
import nest_asyncio
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

nest_asyncio.apply()

generator = TestsetGenerator.from_langchain(
    generator_llm=ChatOpenAI(model="gpt-4o-mini"),
    critic_llm=ChatOpenAI(model="gpt-4o-mini"),
    embeddings=OpenAIEmbeddings(),
)

testset = generator.generate_with_langchain_docs(
    documents,
    test_size=4,
    distributions={simple: 0.5, reasoning: 0.25, multi_context: 0.25},
)

embedding nodes:   0%|          | 0/902 [00:00<?, ?it/s]

Generating:   0%|          | 0/4 [00:00<?, ?it/s]

In [9]:
testset.to_pandas()

,question,contexts,ground_truth,evolution_type,metadata,episode_done
0,What is the purpose of a reference management ...,[# BibTeX\n\n>[BibTeX](https://www.ctan.org/pk...,BibTeX is a reference management system that s...,simple,[{'source': 'docs/docs/integrations/providers/...,True
1,What is the purpose of the Document Loader in ...,[# Telegram\n\n>[Telegram Messenger](https://w...,The answer to given question is not present in...,simple,[{'source': 'docs/docs/integrations/providers/...,True
2,"How are LLM observability, prompt tracking, an...",[# PromptLayer\n\n>[PromptLayer](https://docs....,PromptLayer is a platform that helps with LLM ...,reasoning,[{'source': 'docs/docs/integrations/providers/...,True
3,How does PromptLayer boost prompt engineering ...,[# PromptLayer\n\n>[PromptLayer](https://docs....,The context does not provide specific informat...,multi_context,[{'source': 'docs/docs/integrations/providers/...,True


### LangSmith の Dataset の作成


In [10]:
from langsmith import Client

dataset_name = "agent-book"

client = Client()

if client.has_dataset(dataset_name=dataset_name):
    client.delete_dataset(dataset_name=dataset_name)

dataset = client.create_dataset(dataset_name=dataset_name)

### 合成テストデータの保存


In [11]:
inputs = []
outputs = []
metadatas = []

for testset_record in testset.test_data:
    inputs.append(
        {
            "question": testset_record.question,
        }
    )
    outputs.append(
        {
            "contexts": testset_record.contexts,
            "ground_truth": testset_record.ground_truth,
        }
    )
    metadatas.append(
        {
            "source": testset_record.metadata[0]["source"],
            "evolution_type": testset_record.evolution_type,
        }
    )

In [12]:
client.create_examples(
    inputs=inputs,
    outputs=outputs,
    metadata=metadatas,
    dataset_id=dataset.id,
)

## 7.5. LangSmith と Ragas を使ったオフライン評価の実装


### カスタム Evaluator の実装


In [13]:
from typing import Any

from langchain_core.embeddings import Embeddings
from langchain_core.language_models import BaseChatModel
from langsmith.schemas import Example, Run
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics.base import Metric, MetricWithEmbeddings, MetricWithLLM


class RagasMetricEvaluator:
    def __init__(self, metric: Metric, llm: BaseChatModel, embeddings: Embeddings):
        self.metric = metric

        # LLMとEmbeddingsをMetricに設定
        if isinstance(self.metric, MetricWithLLM):
            self.metric.llm = LangchainLLMWrapper(llm)
        if isinstance(self.metric, MetricWithEmbeddings):
            self.metric.embeddings = LangchainEmbeddingsWrapper(embeddings)

    def evaluate(self, run: Run, example: Example) -> dict[str, Any]:
        context_strs = [doc.page_content for doc in run.outputs["contexts"]]

        # Ragasの評価メトリクスのscoreメソッドでスコアを算出
        score = self.metric.score(
            {
                "question": example.inputs["question"],
                "answer": run.outputs["answer"],
                "contexts": context_strs,
                "ground_truth": example.outputs["ground_truth"],
            },
        )
        return {"key": self.metric.name, "score": score}

In [15]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.metrics import answer_relevancy, context_precision

metrics = [context_precision, answer_relevancy]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

evaluators = [
    RagasMetricEvaluator(metric, llm, embeddings).evaluate
    for metric in metrics
]

### 推論の関数の実装


In [16]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma.from_documents(documents, embeddings)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [17]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template('''\
以下の文脈だけを踏まえて質問に回答してください。

文脈: """
{context}
"""

質問: {question}
''')

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

retriever = db.as_retriever()

chain = RunnableParallel(
    {
        "question": RunnablePassthrough(),
        "context": retriever,
    }
).assign(answer=prompt | model | StrOutputParser())

In [18]:
def predict(inputs: dict[str, Any]) -> dict[str, Any]:
    question = inputs["question"]
    output = chain.invoke(question)
    return {
        "contexts": output["context"],
        "answer": output["answer"],
    }

### オフライン評価の実装・実行


In [19]:
from langsmith.evaluation import evaluate

evaluate(
    predict,
    data="agent-book",
    evaluators=evaluators,
)

View the evaluation results for experiment: 'monthly-news-91' at:
https://smith.langchain.com/o/d864e70f-8bfc-4f11-b774-2962b1ee3f41/datasets/b3738290-1180-44a7-9dd5-d5765fb35116/compare?selectedSessions=deb03bfd-f406-44d9-b962-b487fc90a977




0it [00:00, ?it/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


,inputs.question,outputs.contexts,outputs.answer,error,reference.contexts,reference.ground_truth,feedback.context_precision,feedback.answer_relevancy,execution_time,example_id,id
0,What is the purpose of the Document Loader in ...,[page_content='# Telegram\n\n>[Telegram Messen...,The purpose of the Document Loader in the cont...,None,[# Telegram\n\n>[Telegram Messenger](https://w...,The answer to given question is not present in...,0.0,1.000000,1.949709,5267f721-ef4c-400a-a99c-6866d3c8a138,188e7611-8283-43b9-a0e5-213e6f804700
1,How does PromptLayer boost prompt engineering ...,[page_content='# PromptLayer\n\n>[PromptLayer]...,PromptLayer enhances prompt engineering and LL...,None,[# PromptLayer\n\n>[PromptLayer](https://docs....,The context does not provide specific informat...,0.0,0.862841,2.885015,0e7e12fe-e780-4569-a71f-58cbdc49349c,5e335ca5-6ce1-4c2d-9081-5cfac17cc75a
2,What is the purpose of a reference management ...,[page_content='# BibTeX\n\n>[BibTeX](https://w...,BibTeX is a file format and reference manageme...,None,[# BibTeX\n\n>[BibTeX](https://www.ctan.org/pk...,BibTeX is a reference management system that s...,1.0,0.767896,3.312509,ee5aba8b-c6e9-491e-9bf9-3e2c453226f3,57a645ef-797a-4b00-be79-c6d5aacee47d
3,"How are LLM observability, prompt tracking, an...",[page_content='# PromptLayer\n\n>[PromptLayer]...,"LLM observability, prompt tracking, and Prompt...",None,[# PromptLayer\n\n>[PromptLayer](https://docs....,PromptLayer is a platform that helps with LLM ...,1.0,0.675597,2.320739,abad532d-4e6b-48dd-a970-b94295b7d357,cc9ab83a-f9c9-4edc-8f3f-435d5f546ddb


## LangSmith を使ったオンライン評価の実装


### フィードバックボタンを表示する関数の実装


In [20]:
from uuid import UUID

import ipywidgets as widgets
from IPython.display import display
from langsmith import Client


def display_feedback_buttons(run_id: UUID) -> None:
    # GoodボタンとBadボタンを準備
    good_button = widgets.Button(
        description="Good",
        button_style="success",
        icon="thumbs-up",
    )
    bad_button = widgets.Button(
        description="Bad",
        button_style="danger",
        icon="thumbs-down",
    )

    # クリックされた際に実行される関数を定義
    def on_button_clicked(button: widgets.Button) -> None:
        if button == good_button:
            score = 1
        elif button == bad_button:
            score = 0
        else:
            raise ValueError(f"Unknown button: {button}")

        client = Client()
        client.create_feedback(run_id=run_id, key="thumbs", score=score)
        print("フィードバックを送信しました")

    # ボタンがクリックされた際にon_button_clicked関数を実行
    good_button.on_click(on_button_clicked)
    bad_button.on_click(on_button_clicked)

    # ボタンを表示
    display(good_button, bad_button)

### フィードバックボタンを表示


In [23]:
from langchain_core.tracers.context import collect_runs

# LangSmithのトレースのID(Run ID)を取得するため、collect_runs関数を使用
with collect_runs() as runs_cb:
    output = chain.invoke("LangChainの概要を教えて")
    print(output["answer"])
    run_id = runs_cb.traced_runs[0].id

display_feedback_buttons(run_id)

LangChainは、大規模言語モデル（LLM）を活用したアプリケーションを開発するためのフレームワークです。このフレームワークは、LLMアプリケーションのライフサイクルの各段階を簡素化します。具体的には、以下のような機能を提供しています。

- **開発**: LangChainのオープンソースのビルディングブロックやコンポーネント、サードパーティの統合を使用してアプリケーションを構築します。また、LangGraphを利用して、状態を持つエージェントを構築し、ストリーミングや人間の介入をサポートします。
- **生産化**: LangSmithを使用して、チェーンを検査、監視、評価し、継続的に最適化して自信を持ってデプロイできます。
- **デプロイ**: LangGraphアプリケーションを本番環境向けのAPIやアシスタントに変換します。

LangChainは、以下の主要なオープンソースライブラリで構成されています：

- **`langchain-core`**: 基本的な抽象化とLangChain表現言語（LCEL）。
- **`langchain-community`**: サードパーティの統合。
- **`langchain`**: アプリケーションの認知アーキテクチャを構成するチェーン、エージェント、検索戦略。
- **LangGraph**: LLMを用いた堅牢で状態を持つマルチアクターアプリケーションを構築するためのツール。

LangChainは、アプリケーションの開発からデプロイまでのプロセスを効率化し、開発者がより簡単にLLMアプリケーションを構築できるように設計されています。


Button(button_style='success', description='Good', icon='thumbs-up', style=ButtonStyle())

Button(button_style='danger', description='Bad', icon='thumbs-down', style=ButtonStyle())

フィードバックを送信しました
